# Notebook 01: Data Integration and Preprocessing
**Project:** JobLens Vietnam - Labor Market Analysis & Skill Trends  
**Course:** Introduction to Data Science  

### Objective:
1. Inspect the two raw datasets: `tinixai_it_ai_data_jobs.csv` and `topcv_jobs.csv`.
2. Standardize column schemas, assign global unique IDs (`unique_job_id`), and track source provenance.
3. Perform intra-source and cross-source deduplication with normalized composite keys (`reports/tables/deduplication_audit.csv`).
4. Clean text fields (stripping HTML, unescaping characters, preserving essential technical tokens like `C++`, `SQL`, `Python`, `AWS`).
5. Audit and normalize regional locations with explicit city detection first (`reports/tables/location_fix_audit.csv`).
6. Parse salary expressions into VND/month with independent bounds and disclosure categories (`reports/tables/salary_parsing_audit.csv`).
7. Generate standardized rule-based `job_role_label` categories (weak supervision taxonomy).
8. Export processed subsets for downstream classification, regression, and clustering.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to sys.path
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config as config
from src.data.loader import load_raw_datasets, inspect_dataset, standardize_and_merge, deduplicate_merged
from src.data.cleaner import clean_dataframe, generate_location_audit
from src.data.salary_parser import process_salaries
from src.data.label_generator import generate_labels

## 1. Inspect Raw Datasets
Let's inspect row counts, schema definitions, and missing value profiles for both data sources without mutating raw files.

In [ ]:
df_tinixai, df_topcv = load_raw_datasets()

info_tinixai = inspect_dataset(df_tinixai, "TinixAI IT/AI Jobs")
info_topcv = inspect_dataset(df_topcv, "TopCV IT Jobs")

print("=== TinixAI Dataset ===")
print(f"Shape: {info_tinixai['rows']} rows x {info_tinixai['cols']} columns")
print(f"Missing values:\n{pd.Series(info_tinixai['missing_values'])}")

print("\n=== TopCV Dataset ===")
print(f"Shape: {info_topcv['rows']} rows x {info_topcv['cols']} columns")
print(f"Missing values:\n{pd.Series(info_topcv['missing_values'])}")

## 2. Standardize Schema and Deduplicate
We map both datasets into a common 16-attribute schema, synthesize `unique_job_id`, and remove duplicates using normalized composite keys.

In [ ]:
merged_raw = standardize_and_merge(df_tinixai, df_topcv)
print(f"Raw merged records: {len(merged_raw)}")

merged_deduped, removed_records, dedup_audit = deduplicate_merged(merged_raw)
print(f"Unique postings retained: {len(merged_deduped)} (removed {len(removed_records)} duplicates)")
print("\nDeduplication Audit Breakdown:")
display(dedup_audit) if 'display' in dir() else print(dedup_audit.to_string(index=False))

## 3. Location Normalization and Audit
Fix the previous location normalization bug where Hanoi locations containing 'quận' were erroneously mapped to Ho Chi Minh City.

In [ ]:
loc_audit_df = generate_location_audit(merged_deduped)
print(f"Location audit records affected: {len(loc_audit_df)} unique strings ({loc_audit_df['affected_records'].sum()} postings)")
display(loc_audit_df.head(10)) if 'display' in dir() else print(loc_audit_df.head(10).to_string(index=False))

cleaned_df = clean_dataframe(merged_deduped)
print("\nNormalized Regional Distribution:")
print(cleaned_df['norm_location'].value_counts())

## 4. Salary Parsing and Independent Validation
Parse salaries, categorize disclosure types, and validate bounds independently ($3\text{M} \le \text{min} \le \text{max} \le 150\text{M}$).

In [ ]:
parsed_df, salary_cats, salary_audit = process_salaries(cleaned_df)
print("=== Salary Disclosure Categories ===")
display(salary_cats) if 'display' in dir() else print(salary_cats.to_string(index=False))

print("\n=== Salary Parsing Audit Summary ===")
display(salary_audit) if 'display' in dir() else print(salary_audit.to_string(index=False))

## 5. Rule-Based Job Role Generation (Weak Supervision)
Construct heuristic role pseudo-labels from `job_title` using hierarchical regex rules.

In [ ]:
labeled_df, label_dist, label_mapping = generate_labels(parsed_df)
print("=== Job Role Label Distribution ===")
display(label_dist) if 'display' in dir() else print(label_dist.to_string(index=False))